In [26]:
# Cell 1：基础环境

import torch
import transformers

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Transformers:",
    transformers.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)

PyTorch: 2.8.0+cu128
Transformers: 4.57.6
CUDA available: True
Device: cuda


In [27]:
# Cell 2：读取冻结的数据划分

from pathlib import Path
import pandas as pd
import numpy as np
import random


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


train_df = pd.read_csv(
    PROCESSED_DATA_DIR
    / "train_77.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DATA_DIR
    / "validation_77.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA_DIR
    / "test_77.csv"
)


print(
    "Train:",
    train_df.shape
)

print(
    "Validation:",
    validation_df.shape
)

print(
    "Test:",
    test_df.shape
)

Train: (9149, 2)
Validation: (1961, 2)
Test: (1961, 2)


In [28]:
# Cell 3：77类别映射

class_names = sorted(
    train_df["category"].unique()
)

label_to_id = {
    label: index
    for index, label
    in enumerate(class_names)
}

id_to_label = {
    index: label
    for label, index
    in label_to_id.items()
}

NUM_CLASSES = len(class_names)

print(
    "Number of classes:",
    NUM_CLASSES
)

Number of classes: 77


In [29]:
train_df["label"] = (
    train_df["category"]
    .map(label_to_id)
)

validation_df["label"] = (
    validation_df["category"]
    .map(label_to_id)
)

test_df["label"] = (
    test_df["category"]
    .map(label_to_id)
)

In [30]:
# Cell 4：加载DistilBERT tokenizer

from transformers import AutoTokenizer


MODEL_NAME = (
    "distilbert/distilbert-base-uncased"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    tokenizer
)

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [31]:
# Cell 5：检查DistilBERT分词

sample_text = (
    "My transfer has not arrived yet"
)

tokens = tokenizer.tokenize(
    sample_text
)

token_ids = tokenizer.convert_tokens_to_ids(
    tokens
)

print(
    "Original:"
)

print(
    sample_text
)

print()

print(
    "Tokens:"
)

print(
    tokens
)

print()

print(
    "Token IDs:"
)

print(
    token_ids
)

Original:
My transfer has not arrived yet

Tokens:
['my', 'transfer', 'has', 'not', 'arrived', 'yet']

Token IDs:
[2026, 4651, 2038, 2025, 3369, 2664]


In [32]:
# Cell 6：完整Tokenizer输出

encoded_example = tokenizer(
    sample_text,
    truncation=True,
    return_tensors="pt"
)

for key, value in encoded_example.items():

    print(
        key,
        value
    )

    print(
        "shape:",
        value.shape
    )

    print()

input_ids tensor([[ 101, 2026, 4651, 2038, 2025, 3369, 2664,  102]])
shape: torch.Size([1, 8])

attention_mask tensor([[1, 1, 1, 1, 1, 1, 1, 1]])
shape: torch.Size([1, 8])



In [33]:
# Cell 7：训练集DistilBERT token长度

def get_transformer_length(
    text
):

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )

    return len(
        encoded["input_ids"]
    )


transformer_train_lengths = (
    train_df["text"]
    .apply(
        get_transformer_length
    )
)


print(
    transformer_train_lengths.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    9149.000000
mean       15.963821
std         9.028020
min         4.000000
50%        13.000000
75%        17.000000
90%        27.000000
95%        35.000000
99%        52.000000
max        98.000000
Name: text, dtype: float64


In [34]:
# Cell 8：转换为Hugging Face Dataset

from datasets import Dataset


train_dataset = Dataset.from_pandas(
    train_df[
        ["text", "label"]
    ],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        ["text", "label"]
    ],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[
        ["text", "label"]
    ],
    preserve_index=False
)

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 9149
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1961
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1961
})


In [35]:
# Cell 9：Transformer tokenization

MAX_LENGTH = 64


def tokenize_batch(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [36]:
# Cell 10：Tokenize，并删除原始text字符串列

tokenized_train_dataset = (
    train_dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"]
    )
)

tokenized_validation_dataset = (
    validation_dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"]
    )
)

tokenized_test_dataset = (
    test_dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"]
    )
)

Map: 100%|██████████| 1961/1961 [00:00<00:00, 108942.24 examples/s]


In [37]:
# Cell 11：动态Padding

from transformers import (
    DataCollatorWithPadding
)


data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)

print(data_collator)

DataCollatorWithPadding(tokenizer=DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tenso

In [38]:
# Cell 12：加载预训练DistilBERT分类模型

from transformers import (
    AutoModelForSequenceClassification
)


model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        id2label=id_to_label,
        label2id=label_to_id
    )
)

model = model.to(device)

print(model)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [39]:
# Cell 13：测试动态Padding和模型Forward

sample_batch = [
    tokenized_train_dataset[i]
    for i in range(4)
]

batch = data_collator(
    sample_batch
)

print(
    "input_ids:",
    batch["input_ids"].shape
)

print(
    "attention_mask:",
    batch["attention_mask"].shape
)

print(
    "labels:",
    batch["labels"].shape
)

input_ids: torch.Size([4, 27])
attention_mask: torch.Size([4, 27])
labels: torch.Size([4])


In [40]:
batch_on_device = {
    key: value.to(device)
    for key, value in batch.items()
}

with torch.no_grad():

    outputs = model(
        **batch_on_device
    )

print(
    "Logits shape:",
    outputs.logits.shape
)

print(
    "Loss:",
    outputs.loss.item()
)

Logits shape: torch.Size([4, 77])
Loss: 4.385970115661621


In [41]:
# Cell 14：定义Transformer评价指标

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1
    }

In [42]:
# Cell 15：Transformer训练配置

from transformers import TrainingArguments


training_args = TrainingArguments(

    output_dir=(
        PROJECT_ROOT
        / "models"
        / "distilbert_baseline"
    ),

    # -------------------------
    # Epoch
    # -------------------------

    num_train_epochs=10,

    # -------------------------
    # Batch size
    # -------------------------

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    # -------------------------
    # Optimizer
    # -------------------------

    learning_rate=2e-5,

    weight_decay=0.01,

    # -------------------------
    # Evaluation / Saving
    # -------------------------

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    save_total_limit=2,

    # -------------------------
    # Logging
    # -------------------------

    logging_strategy="steps",

    logging_steps=100,

    # -------------------------
    # Reproducibility
    # -------------------------

    seed=SEED,

    data_seed=SEED,

    # -------------------------
    # Reporting
    # -------------------------

    report_to="none"
)

In [43]:
# Cell 16：创建Trainer

from transformers import Trainer


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=(
        tokenized_train_dataset
    ),

    eval_dataset=(
        tokenized_validation_dataset
    ),

    data_collator=data_collator,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

In [44]:
# Cell 17：Fine-tune DistilBERT

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,2.475500,1.895906,0.707802,0.718265,0.686148,0.657426
2,0.949200,0.802496,0.860275,0.872793,0.855872,0.856835
3,0.451900,0.473826,0.898011,0.904381,0.897486,0.898244
4,0.282700,0.377657,0.907700,0.914343,0.907119,0.908002
5,0.167300,0.342262,0.912290,0.916437,0.911622,0.912020
6,0.109000,0.355822,0.909230,0.915563,0.910140,0.910314
7,0.057000,0.352761,0.913310,0.919478,0.913344,0.914009
8,0.042500,0.355578,0.915859,0.920248,0.915493,0.915725
9,0.039900,0.361885,0.915859,0.920462,0.915219,0.915490
10,0.036900,0.356288,0.913819,0.918158,0.913210,0.913603


In [45]:
# Cell 18：查看最佳checkpoint

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation metric:",
    trainer.state.best_metric
)

Best checkpoint: c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\models\distilbert_baseline\checkpoint-4576
Best validation metric: 0.9157246038214969


In [46]:
# Cell 19：确认最佳checkpoint

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation Macro-F1:",
    trainer.state.best_metric
)

Best checkpoint: c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\models\distilbert_baseline\checkpoint-4576
Best validation Macro-F1: 0.9157246038214969


In [47]:
# Cell 20：DistilBERT最终测试集评估

test_results = trainer.evaluate(
    eval_dataset=tokenized_test_dataset,
    metric_key_prefix="test"
)

print("DistilBERT Test Results")
print("-" * 40)

print(
    "Test Loss:",
    f"{test_results['test_loss']:.4f}"
)

print(
    "Test Accuracy:",
    f"{test_results['test_accuracy']:.4f}"
)

print(
    "Test Macro Precision:",
    f"{test_results['test_macro_precision']:.4f}"
)

print(
    "Test Macro Recall:",
    f"{test_results['test_macro_recall']:.4f}"
)

print(
    "Test Macro-F1:",
    f"{test_results['test_macro_f1']:.4f}"
)

DistilBERT Test Results
----------------------------------------
Test Loss: 0.3325
Test Accuracy: 0.9240
Test Macro Precision: 0.9295
Test Macro Recall: 0.9262
Test Macro-F1: 0.9258


In [48]:
# Cell 21：获得测试集预测

test_prediction_output = trainer.predict(
    tokenized_test_dataset
)

test_logits = (
    test_prediction_output.predictions
)

test_label_ids = (
    test_prediction_output.label_ids
)

test_pred_ids = np.argmax(
    test_logits,
    axis=-1
)

print(
    "Logits shape:",
    test_logits.shape
)

print(
    "Predictions:",
    test_pred_ids.shape
)

print(
    "Labels:",
    test_label_ids.shape
)

Logits shape: (1961, 77)
Predictions: (1961,)
Labels: (1961,)


In [49]:
# Cell 22：建立DistilBERT测试结果DataFrame

distilbert_test_df = (
    test_df
    .reset_index(drop=True)
    .copy()
)

distilbert_test_df[
    "distilbert_prediction"
] = [
    id_to_label[
        int(pred_id)
    ]
    for pred_id in test_pred_ids
]

distilbert_test_df[
    "distilbert_correct"
] = (
    distilbert_test_df["category"]
    ==
    distilbert_test_df[
        "distilbert_prediction"
    ]
)

display(
    distilbert_test_df[
        [
            "text",
            "category",
            "distilbert_prediction",
            "distilbert_correct"
        ]
    ].head(10)
)

,text,category,distilbert_prediction,distilbert_correct
0,Will my transfer immediately show up in my acc...,transfer_timing,balance_not_updated_after_bank_transfer,False
1,I don't have my access code for the app.,passcode_forgotten,passcode_forgotten,True
2,How long do bank transfers take to clear,balance_not_updated_after_bank_transfer,balance_not_updated_after_bank_transfer,True
3,Are there any fees if I want to add money usin...,top_up_by_card_charge,top_up_by_card_charge,True
4,How low does the balance have to be before the...,automatic_top_up,transfer_timing,False
5,Help me figure out how to link the new card.,card_linking,card_linking,True
6,Contactless payments need to be enabled for my...,contactless_not_working,contactless_not_working,True
7,How do I cancel a recent transfer?,cancel_transfer,cancel_transfer,True
8,My contactless stopped working today no matter...,contactless_not_working,contactless_not_working,True
9,How do I deactivate my account?,terminate_account,terminate_account,True


In [50]:
# Cell 23：DistilBERT测试集分类报告

from sklearn.metrics import (
    classification_report
)


distilbert_classification_report = (
    classification_report(
        test_label_ids,
        test_pred_ids,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print(
    distilbert_classification_report
)

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     1.0000    0.9667    0.9831        30
                                activate_my_card     0.9310    0.9000    0.9153        30
                                       age_limit     1.0000    0.9545    0.9767        22
                         apple_pay_or_google_pay     1.0000    1.0000    1.0000        25
                                     atm_support     0.9000    0.9474    0.9231        19
                                automatic_top_up     1.0000    0.9600    0.9796        25
         balance_not_updated_after_bank_transfer     0.8710    0.8438    0.8571        32
balance_not_updated_after_cheque_or_cash_deposit     0.9118    0.9394    0.9254        33
                         beneficiary_not_allowed     0.9375    1.0000    0.9677        30
                                 cancel_transfer     1.0000    1.0000    1.0000        29
         

In [51]:
# Cell 24：保存DistilBERT每类指标

distilbert_report_dict = (
    classification_report(
        test_label_ids,
        test_pred_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
)

distilbert_per_class_df = (
    pd.DataFrame(
        distilbert_report_dict
    )
    .T
)

distilbert_per_class_df = (
    distilbert_per_class_df
    .loc[class_names]
    .reset_index()
    .rename(
        columns={
            "index": "category"
        }
    )
)

display(
    distilbert_per_class_df.head()
)

,category,precision,recall,f1-score,support
0,Refund_not_showing_up,1.000000,0.966667,0.983051,30.0
1,activate_my_card,0.931034,0.900000,0.915254,30.0
2,age_limit,1.000000,0.954545,0.976744,22.0
3,apple_pay_or_google_pay,1.000000,1.000000,1.000000,25.0
4,atm_support,0.900000,0.947368,0.923077,19.0


In [52]:
# Cell 25：保存最终DistilBERT模型

FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "distilbert_final"
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print(
    "Final model saved to:",
    FINAL_MODEL_DIR
)

Final model saved to: c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\models\distilbert_final


In [54]:
# Cell 26：保存DistilBERT测试预测

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

distilbert_test_df.to_csv(
    RESULTS_DIR
    / "distilbert_test_predictions.csv",
    index=False
)

distilbert_per_class_df.to_csv(
    RESULTS_DIR
    / "distilbert_per_class_metrics.csv",
    index=False
)

print(
    "Results saved."
)

Results saved.


In [55]:
# Cell 27：DistilBERT总体错误数量

distilbert_errors_df = (
    distilbert_test_df[
        ~distilbert_test_df["distilbert_correct"]
    ]
    .copy()
)

num_test_samples = len(
    distilbert_test_df
)

num_errors = len(
    distilbert_errors_df
)

error_rate = (
    num_errors
    / num_test_samples
)

print(
    "Test samples:",
    num_test_samples
)

print(
    "Errors:",
    num_errors
)

print(
    "Error rate:",
    f"{error_rate:.4%}"
)

Test samples: 1961
Errors: 149
Error rate: 7.5982%


In [56]:
# Cell 28：查看DistilBERT错误样本

display(
    distilbert_errors_df[
        [
            "text",
            "category",
            "distilbert_prediction"
        ]
    ]
    .head(30)
)

,text,category,distilbert_prediction
0,Will my transfer immediately show up in my acc...,transfer_timing,balance_not_updated_after_bank_transfer
4,How low does the balance have to be before the...,automatic_top_up,transfer_timing
16,"So much time has gone by, the pending status s...",pending_cash_withdrawal,pending_card_payment
69,HOW LONG TO TAKE THE TIME TO SOLVE,card_not_working,card_delivery_estimate
117,what is the leadtime for a new card to show up...,card_delivery_estimate,card_arrival
131,Is it okay to use a bank card to top up,top_up_by_card_charge,topping_up_by_card
147,Can I make a withdraw if I am out of country f...,wrong_exchange_rate_for_cash_withdrawal,atm_support
156,As far as courtries go which ones are supported?,country_support,supported_cards_and_currencies
184,Can you tell me how long it takes for a new ca...,card_delivery_estimate,card_arrival
244,What cards do you offer?,visa_or_mastercard,supported_cards_and_currencies


In [57]:
# Cell 29：统计错误类别组合

distilbert_confusion_pairs_df = (
    distilbert_errors_df
    .groupby(
        [
            "category",
            "distilbert_prediction"
        ]
    )
    .size()
    .reset_index(
        name="error_count"
    )
    .sort_values(
        by="error_count",
        ascending=False
    )
)

display(
    distilbert_confusion_pairs_df.head(20)
)

,category,distilbert_prediction,error_count
8,balance_not_updated_after_bank_transfer,transfer_timing,4
70,pending_transfer,pending_card_payment,4
66,pending_top_up,top_up_failed,4
43,direct_debit_payment_not_recognised,card_payment_fee_charged,3
31,cash_withdrawal_not_recognised,compromised_card,3
41,declined_transfer,failed_transfer,3
106,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate,3
98,unable_to_verify_identity,verify_my_identity,3
83,top_up_failed,top_up_reverted,3
80,top_up_by_cash_or_cheque,topping_up_by_card,3


In [58]:
# Cell 30：每个真实类别的错误数量

errors_by_true_category_df = (
    distilbert_errors_df
    .groupby("category")
    .size()
    .reset_index(
        name="error_count"
    )
    .sort_values(
        by="error_count",
        ascending=False
    )
)

display(
    errors_by_true_category_df.head(15)
)

,category,error_count
13,card_payment_wrong_exchange_rate,7
33,pending_top_up,6
22,direct_debit_payment_not_recognised,6
39,top_up_failed,6
34,pending_transfer,6
21,declined_transfer,5
50,wrong_exchange_rate_for_cash_withdrawal,5
16,cash_withdrawal_not_recognised,5
12,card_payment_not_recognised,5
5,balance_not_updated_after_bank_transfer,5


In [59]:
# Cell 31：DistilBERT表现最差的类别

worst_distilbert_classes_df = (
    distilbert_per_class_df
    .sort_values(
        by="f1-score",
        ascending=True
    )
    .head(15)
)

display(
    worst_distilbert_classes_df
)

,category,precision,recall,f1-score,support
23,compromised_card,0.714286,0.833333,0.769231,18.0
59,top_up_failed,0.814815,0.785714,0.800000,28.0
62,topping_up_by_card,0.714286,0.909091,0.800000,22.0
67,transfer_timing,0.777778,0.840000,0.807692,25.0
48,pending_top_up,0.851852,0.793103,0.821429,29.0
18,card_payment_wrong_exchange_rate,0.888889,0.774194,0.827586,31.0
66,transfer_not_received_by_recipient,0.823529,0.875000,0.848485,32.0
17,card_payment_not_recognised,0.866667,0.838710,0.852459,31.0
76,wrong_exchange_rate_for_cash_withdrawal,0.866667,0.838710,0.852459,31.0
36,failed_transfer,0.800000,0.923077,0.857143,26.0


In [60]:
# Cell 32：Recall最低类别

lowest_recall_df = (
    distilbert_per_class_df
    .sort_values(
        by="recall",
        ascending=True
    )
    .head(10)
)

display(
    lowest_recall_df[
        [
            "category",
            "precision",
            "recall",
            "f1-score",
            "support"
        ]
    ]
)

,category,precision,recall,f1-score,support
18,card_payment_wrong_exchange_rate,0.888889,0.774194,0.827586,31.0
59,top_up_failed,0.814815,0.785714,0.800000,28.0
49,pending_transfer,0.956522,0.785714,0.862745,28.0
48,pending_top_up,0.851852,0.793103,0.821429,29.0
28,declined_transfer,0.954545,0.807692,0.875000,26.0
24,contactless_not_working,1.000000,0.818182,0.900000,11.0
29,direct_debit_payment_not_recognised,0.903226,0.823529,0.861538,34.0
13,card_delivery_estimate,0.904762,0.826087,0.863636,23.0
15,card_not_working,0.950000,0.826087,0.883721,23.0
23,compromised_card,0.714286,0.833333,0.769231,18.0


In [61]:
# Cell 33：Precision最低类别

lowest_precision_df = (
    distilbert_per_class_df
    .sort_values(
        by="precision",
        ascending=True
    )
    .head(10)
)

display(
    lowest_precision_df[
        [
            "category",
            "precision",
            "recall",
            "f1-score",
            "support"
        ]
    ]
)

,category,precision,recall,f1-score,support
23,compromised_card,0.714286,0.833333,0.769231,18.0
62,topping_up_by_card,0.714286,0.909091,0.800000,22.0
67,transfer_timing,0.777778,0.840000,0.807692,25.0
36,failed_transfer,0.800000,0.923077,0.857143,26.0
46,pending_card_payment,0.800000,0.933333,0.861538,30.0
69,verify_my_identity,0.807692,1.000000,0.893617,21.0
42,lost_or_stolen_card,0.809524,0.944444,0.871795,18.0
59,top_up_failed,0.814815,0.785714,0.800000,28.0
66,transfer_not_received_by_recipient,0.823529,0.875000,0.848485,32.0
16,card_payment_fee_charged,0.850000,1.000000,0.918919,34.0


In [62]:
# Cell 34：查看最常见错误组合对应的文本

top_confusion_pairs = (
    distilbert_confusion_pairs_df
    .head(10)
)

for _, row in (
    top_confusion_pairs.iterrows()
):

    true_category = row[
        "category"
    ]

    predicted_category = row[
        "distilbert_prediction"
    ]

    pair_examples = (
        distilbert_errors_df[
            (
                distilbert_errors_df[
                    "category"
                ]
                == true_category
            )
            &
            (
                distilbert_errors_df[
                    "distilbert_prediction"
                ]
                == predicted_category
            )
        ]
    )

    print(
        "=" * 90
    )

    print(
        "TRUE:",
        true_category
    )

    print(
        "PREDICTED:",
        predicted_category
    )

    print(
        "COUNT:",
        len(pair_examples)
    )

    print()

    for text in (
        pair_examples["text"]
        .head(5)
    ):

        print(
            "-",
            text
        )

TRUE: balance_not_updated_after_bank_transfer
PREDICTED: transfer_timing
COUNT: 4

- How long does it take for money to reach my account?
- When will my transfer process?
- Can you check if my money has reached my account since it is not showing up yet?
- When will my transfer be available in my account.
TRUE: pending_transfer
PREDICTED: pending_card_payment
COUNT: 4

- I have a trasnger that is pending and wondered about it.
- I have a pending transaction
- Can you explain why my payment is still pending?
- What is the average pending time for a transaction?
TRUE: pending_top_up
PREDICTED: top_up_failed
COUNT: 4

- I added money to the card but the transaction didn't go through
- I tried to top up, but it didn't finish.
- Even though I added money ey to the card, I couldn't use it
- Is my top up not working?
TRUE: direct_debit_payment_not_recognised
PREDICTED: card_payment_fee_charged
COUNT: 3

- Someone else charged my card!
- i do not know what this charge is for
- what is this char

In [63]:
# Cell 35：保存DistilBERT错误分析

distilbert_errors_df.to_csv(
    RESULTS_DIR
    / "distilbert_test_errors.csv",
    index=False
)

distilbert_confusion_pairs_df.to_csv(
    RESULTS_DIR
    / "distilbert_confusion_pairs.csv",
    index=False
)

print(
    "DistilBERT error analysis saved."
)

DistilBERT error analysis saved.
